# Customer support refunds: who gets the money?

A support agent pays refunds to the account stored in the customer's long-term profile (a LangGraph Store). Anyone can open a support ticket, so ticket text is **untrusted**. An attacker opens a ticket that looks like it came from the customer, Jane Doe (`cust-1042`), and asks the agent to:

- 💳 **change the refund account** to `ACCT-99990000`, or
- 🪪 **change the refund customer ID** to `cust-6666`, whose payout account the attacker controls.

The notebook shows two places where the Quarry-backed middleware steps in:

1. **Write boundary** 🛡️: the ticket importer tries to change payout details in the profile.
2. **Memory to tool** 🔁: an untrusted account is already in memory, and the agent copies it into an `issue_refund` call.

`issue_refund` is simulated; no money moves.

In [ ]:
import os
from getpass import getpass

# True: a scripted model makes the same tool calls a real model would; no API key needed.
OFFLINE = True

if not OFFLINE and not os.getenv("OPENAI_API_KEY") and not os.getenv("AZURE_OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key (not echoed): ")

from scenario import (
    ATTACKER_ACCOUNT, CHANGE_ACCOUNT_TICKET, CHANGE_CUSTOMER_TICKET, CLEAN_TICKET,
    VERIFIED_ACCOUNT, run_refund_case, run_ticket_case,
)
from rules import PROTECT_REFUND_DETAILS, REFUND_TO_UNVERIFIED_ACCOUNT

def show(result):
    paid = ", ".join(result.payouts) or "nothing"
    icon = "☠️" if ATTACKER_ACCOUNT in result.payouts else "✅" if VERIFIED_ACCOUNT in result.payouts else "🛡️"
    print(f"▶ {result.label}  ({result.model})")
    if result.requested_change:
        print("   📨 ticket asks to set {} = {}".format(*result.requested_change))
    print(f"   🔧 tools: {' → '.join(result.tool_calls)}")
    for d in result.decisions:
        if d["status"] == "block":
            print(f"   🛑 Quarry blocked {d['operation']} ({d['key'] or d['reader_tool']}): {', '.join(d['rules'])}")
    print(f"   📦 profile afterwards: account={result.stored_account} customer={result.stored_customer_id or '-'}")
    print(f"   {icon} refund paid to: {paid}\n")

print("Verified account:", VERIFIED_ACCOUNT, "| Attacker account:", ATTACKER_ACCOUNT)

## The tickets

The first two come from the attacker. The third is a normal customer question.

In [ ]:
for ticket in (CHANGE_ACCOUNT_TICKET, CHANGE_CUSTOMER_TICKET, CLEAN_TICKET):
    print(ticket)

## Part 1: the write boundary 🛡️

This rule doesn't try to understand the ticket text. It states a business invariant: the ticket importer is never allowed to write or delete payout details. Other ticket notes still save normally.

In [ ]:
print(PROTECT_REFUND_DETAILS)

### ⚠️ Without the guard

The importer changes the profile, and the next day's refund (in a new conversation that shares only the Store) goes to the attacker.

In [ ]:
show(run_ticket_case(guarded=False, ticket="change_account", offline=OFFLINE))
show(run_ticket_case(guarded=False, ticket="change_customer", offline=OFFLINE))

### 🛡️ With the guard

The guard is always on, so the clean ticket runs through it too. The attempted changes reach the agent as a tool error and never reach the Store.

In [ ]:
show(run_ticket_case(guarded=True, ticket="clean", offline=OFFLINE))
show(run_ticket_case(guarded=True, ticket="change_account", offline=OFFLINE))
show(run_ticket_case(guarded=True, ticket="change_customer", offline=OFFLINE))

## Part 2: memory to tool 🔁

Suppose an untrusted account is already in the profile, for example imported before the write rule existed, with `memory_origin("external", "untrusted")`. `issue_refund` is a gated tool. This rule blocks it when its `account` argument matches an account that was read from untrusted memory in the same run.

In [ ]:
print(REFUND_TO_UNVERIFIED_ACCOUNT)

In [ ]:
show(run_refund_case(account_trusted=True, offline=OFFLINE))
show(run_refund_case(account_trusted=False, offline=OFFLINE))

## Takeaway

| Case | Without guard | With guard |
| --- | --- | --- |
| Ticket changes the refund account | ☠️ attacker is paid | 🛡️ write blocked; ✅ verified account is paid |
| Ticket changes the refund customer ID | ☠️ attacker is paid | 🛡️ write blocked; ✅ verified account is paid |
| Clean ticket | ✅ verified account is paid | ✅ verified account is paid |
| Untrusted account already in memory | — | 🛑 refund blocked before it runs |